# GPFDIST — 30 заданий

GPFDIST передаёт файлы сегментам Greenplum по HTTP. Курс использует `gpfdist://cdw:8080`, существующий `countries.csv` и отдельные учебные файлы. Рабочие объекты — только `m_razhin`.

## Результаты обучения

После **GPFDIST** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

gpfdist — параллельный HTTP transport: сегменты читают разные части внешнего файла. External table хранит metadata формата и LOCATION, а reject limit определяет отношение к плохим строкам.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

countries.csv и учебные CSV/PSV файлы. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Проверьте доступность URI из segment network, delimiter/quote/header/encoding, профиль reject и reconciliation external→internal.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Зачем GPFDIST

Обычный клиентский INSERT пропускает данные через одно соединение coordinator. GPFDIST позволяет segment processes параллельно читать части файла или писать выходные фрагменты. Это транспорт, а не формат хранения и не база данных.

## 2. Архитектура

```text
files → gpfdist HTTP server ← segment processes
                         ↑
                 external table metadata
```

Coordinator хранит определение external table и планирует запрос. Сегменты обращаются к URI. Поэтому имя host должно разрешаться из Docker-сети сегментов, а порт — быть доступен им, не только Windows.

## 3. Readable external table

Readable external table описывает колонки, LOCATION, FORMAT, encoding и error policy. Она не копирует данные при CREATE. Каждый SELECT заново читает источник. Изменение файла меняет результат без DDL.

## 4. CSV parsing

Delimiter разделяет поля вне quotes. QUOTE позволяет включить delimiter/newline в значение. ESCAPE кодирует quote/escape. HEADER пропускает первую строку каждого файла. NULL marker задаёт специальное текстовое представление отсутствующего значения.

## 5. Сначала raw text

Надёжный ingestion часто разделяет raw external с текстовыми колонками и typed staging. Если сразу объявить integer/date, одна некорректная строка становится parsing error до выполнения вашего SELECT.

## 6. Reject handling

`SEGMENT REJECT LIMIT` разрешает пропустить ограниченное число или процент ошибочных строк на сегменте. Это не означает «игнорировать качество»: rejected rows должны учитываться, диагностироваться и иметь согласованный порог остановки.

### Почему limit сегментный

Ошибка считается там, где строка обрабатывается. При неравномерном распределении входных фрагментов один сегмент может превысить limit раньше общего ожидаемого числа. Поэтому анализируют и абсолютное количество, и распределение ошибок.

## 7. Внутренняя target

External table подходит для ingress, но регулярные JOIN обычно выполняют по внутренней AO/heap target с выбранной distribution policy, статистикой и quality constraints. Этапы: external raw → profile → typed staging → target → reconciliation.

## 8. Несколько LOCATION

Несколько URI увеличивают параллелизм и позволяют читать файловые shards. Все фрагменты обязаны иметь совместимые формат и схему. HEADER будет применён к каждому URI, что правильно только если каждый shard имеет собственный header.

## 9. Writable external

INSERT в writable external отправляет строки GPFDIST, который создаёт выходные файлы/фрагменты. Повторный INSERT не является автоматическим overwrite. Идемпотентность экспорта требует управления каталогом и именем выгрузки.

## 10. Производительность

На скорость влияют число и размер файлов, ширина строк, parsing, сеть, число сегментов и последующая target policy. Много крошечных файлов создаёт overhead; один огромный источник может ограничить параллелизм.

## 11. Диагностика

Проверяйте процесс/порт GPFDIST, DNS host из сегмента, LOCATION, права на каталог, логи сервера, FORMAT и encoding. Ошибка `connection refused` отличается от parse error: первая до чтения данных, вторая после получения bytes.

## 12. Безопасность

GPFDIST предоставляет файлы из заданного server directory. Не запускайте его от root и не публикуйте каталог с секретами. В учебном стенде endpoint доступен внутри Docker-сети; это не production security model.

## 13. Идемпотентная загрузка

Повтор запуска должен давать тот же target. Для маленького справочника допустим transaction + truncate/insert. Для больших периодов — staging, проверки и exchange/delete+insert по slice. Всегда сохраняйте audit counts.

## 14. Reconciliation

Минимум сравнивают source count, accepted count, rejected count и target count. Для содержимого используют агрегированный checksum по каноническому представлению колонок. Суммы/минимумы полезны, но не доказывают полное равенство.

## 15. Порядок практики

1. Проверить endpoint. 2. Создать raw external. 3. Посмотреть строки и профиль. 4. Настроить error policy. 5. Загрузить staging. 6. Выполнить quality checks. 7. Загрузить target. 8. ANALYZE. 9. Reconcile. 10. Проверить повторный запуск.

### Задание 1. `m_razhin.gpg_01_endpoint`

**Что сделать:** Создайте VIEW с host, port, path и ожидаемым URL учебного GPFDIST.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Источник уже работает на cdw:8080.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_01_endpoint здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',1);

### Задание 2. `m_razhin.gpg_02_countries_ext`

**Что сделать:** Создайте readable external table над countries.csv.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Файл без header, разделитель запятая.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_02_countries_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',2);

### Задание 3. `m_razhin.gpg_03_source_count`

**Что сделать:** Создайте VIEW количества строк внешней таблицы.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Ожидается 173 строки.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_03_source_count здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',3);

### Задание 4. `m_razhin.gpg_04_source_profile`

**Что сделать:** Профилируйте NULL, distinct codes/names и длины.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Внешнюю таблицу сначала исследуют.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_04_source_profile здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',4);

### Задание 5. `m_razhin.gpg_05_countries_heap`

**Что сделать:** Загрузите источник во внутреннюю heap-таблицу.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

INSERT SELECT из external.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_05_countries_heap здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',5);

### Задание 6. `m_razhin.gpg_06_countries_ao`

**Что сделать:** Загрузите в AO column zstd.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Для маленькой таблицы выигрыш не обязателен.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_06_countries_ao здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',6);

### Задание 7. `m_razhin.gpg_07_countries_repl`

**Что сделать:** Создайте replicated-справочник.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Проверяется policy и логический count.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_07_countries_repl здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',7);

### Задание 8. `m_razhin.gpg_08_trim`

**Что сделать:** Создайте очищенную таблицу с trim строк.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Не скрывайте пустые значения как NULL без правила.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_08_trim здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',8);

### Задание 9. `m_razhin.gpg_09_constraints`

**Что сделать:** Создайте quality VIEW дублей и некорректных кодов.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

External table не заменяет quality checks.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_09_constraints здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',9);

### Задание 10. `m_razhin.gpg_10_reconciliation`

**Что сделать:** Сверьте source и target count/checksum.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Идемпотентность начинается с reconciliation.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_10_reconciliation здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',10);

## Уровень 2 — CSV и rejected rows

### Задание 11. `m_razhin.gpg_11_header_ext`

**Что сделать:** Создайте external table для CSV с HEADER.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

HEADER относится к каждому URI-фрагменту.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_11_header_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',11);

### Задание 12. `m_razhin.gpg_12_delimiter_ext`

**Что сделать:** Создайте external table с нестандартным delimiter.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Явно задайте FORMAT options.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_12_delimiter_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',12);

### Задание 13. `m_razhin.gpg_13_quote_escape`

**Что сделать:** Обработайте delimiter внутри quoted field.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

QUOTE и ESCAPE — разные параметры.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_13_quote_escape здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',13);

### Задание 14. `m_razhin.gpg_14_null_mapping`

**Что сделать:** Настройте представление NULL в CSV.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Пустая строка и NULL семантически различаются.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_14_null_mapping здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',14);

### Задание 15. `m_razhin.gpg_15_encoding`

**Что сделать:** Зафиксируйте encoding внешней таблицы.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Несовпадение кодировки проявляется при parsing.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_15_encoding здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',15);

### Задание 16. `m_razhin.gpg_16_bad_rows`

**Что сделать:** Создайте внешний источник с контролируемыми ошибками.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Используйте учебный bad CSV.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_16_bad_rows здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',16);

### Задание 17. `m_razhin.gpg_17_reject_limit`

**Что сделать:** Настройте SEGMENT REJECT LIMIT.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Одна плохая строка не должна отменять допустимую загрузку.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_17_reject_limit здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',17);

### Задание 18. `m_razhin.gpg_18_error_log`

**Что сделать:** Создайте VIEW диагностики rejected rows.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Сохраните raw line и причину.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_18_error_log здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',18);

### Задание 19. `m_razhin.gpg_19_reject_threshold`

**Что сделать:** Продемонстрируйте превышение reject limit безопасно.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Эксперимент выполняйте на учебном файле.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_19_reject_threshold здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',19);

### Задание 20. `m_razhin.gpg_20_type_conversion`

**Что сделать:** Загрузите typed staging с безопасной конверсией.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Сырые строки и typed слой разделяются.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_20_type_conversion здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',20);

## Уровень 3 — parallel locations, export и pipeline

### Задание 21. `m_razhin.gpg_21_multi_location`

**Что сделать:** Создайте external table с несколькими LOCATION.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Каждый URI должен иметь совместимую схему.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_21_multi_location здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',21);

### Задание 22. `m_razhin.gpg_22_parallel_profile`

**Что сделать:** Создайте VIEW строк по gp_segment_id при чтении.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Покажите участие сегментов.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_22_parallel_profile здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',22);

### Задание 23. `m_razhin.gpg_23_distribution_target`

**Что сделать:** Выберите policy внутренней target после GPFDIST.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Внешний источник сам не задаёт policy target.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_23_distribution_target здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',23);

### Задание 24. `m_razhin.gpg_24_writable_ext`

**Что сделать:** Создайте writable external table для выгрузки CSV.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Writable external table используется через INSERT.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_24_writable_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',24);

### Задание 25. `m_razhin.gpg_25_export`

**Что сделать:** Выгрузите агрегат в GPFDIST и сохраните count.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Повторная запись может дописать файлы.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_25_export здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',25);

### Задание 26. `m_razhin.gpg_26_export_readback`

**Что сделать:** Создайте readable external над экспортом.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Round-trip проверяет формат.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_26_export_readback здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',26);

### Задание 27. `m_razhin.gpg_27_roundtrip`

**Что сделать:** Сверьте исходный агрегат и readback checksum.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Count недостаточно для содержимого.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_27_roundtrip здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',27);

### Задание 28. `m_razhin.gpg_28_idempotent_load`

**Что сделать:** Реализуйте повторяемую загрузку countries без дублей.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Staging + replace/merge pattern.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_28_idempotent_load здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',28);

### Задание 29. `m_razhin.gpg_29_audit`

**Что сделать:** Создайте audit VIEW: source,target,rejected,status.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Статус основан на измерениях.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_29_audit здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',29);

### Задание 30. `m_razhin.gpg_30_pipeline`

**Что сделать:** Создайте итоговый VIEW всех этапов GPFDIST pipeline.

Перед DDL запишите ожидаемый формат и failure mode. После — проверьте count, quality и системный каталог.

<details><summary>Подсказка</summary>

Endpoint→external→quality→target→reconciliation.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpg_30_pipeline здесь.

In [ ]:
%%sql
-- Ручная проверка external/target/audit.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('gpfdist',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='gpfdist' ORDER BY task_no;